
# Short Demo Guide

This notebook gives a compact walkthrough of two experiment scripts:

- `experiments/sda-bench.py` compares **Simple Dual Averaging (SDA)** with the **projected subgradient method** on built-in objective functions.
- `experiments/sda-logreg.py` runs **SDA**, **projected subgradient**, and an **sklearn logistic-regression baseline** on CSV datasets.



In [1]:
!python experiments/sda-bench.py --help

usage: sda-bench.py [-h]
                    -o {abs_a2,linf_shift_2d,max_affine_1d,max_affine_4d,weighted_l1_shift_1d,weighted_l1_shift_2d,weighted_l1_shift_4d}
                    -d D -g GAMMA_MULT --alpha ALPHA [--eps EPS] [-i MAX_ITER]
                    [--output-dir OUTPUT_DIR] [-r]

Run Experiment 1 for Simple Dual Averaging and the projected subgradient
method on registry objectives.

options:
  -h, --help            show this help message and exit
  -o, --objective {abs_a2,linf_shift_2d,max_affine_1d,max_affine_4d,weighted_l1_shift_1d,weighted_l1_shift_2d,weighted_l1_shift_4d}
                        Objective registry ID. Available values: abs_a2,
                        linf_shift_2d, max_affine_1d, max_affine_4d,
                        weighted_l1_shift_1d, weighted_l1_shift_2d,
                        weighted_l1_shift_4d.
  -d, --D D             Restriction parameter value to test.
  -g, --gamma-mult GAMMA_MULT
                        Gamma multiplier of the theoretica

In [2]:
!python experiments/sda-logreg.py --help

usage: sda-logreg.py [-h] -d D [-g GAMMA_MULT] --alpha ALPHA [--eps EPS]
                     [-i MAX_ITER] [--lasso] [--lambda LASSO_LAMBDA]
                     [--seed SEED] [--test-size TEST_SIZE]
                     [--output-dir OUTPUT_DIR] [-r]
                     dataset

Run SDA, projected subgradient, and sklearn logistic regression from a CSV
dataset.

positional arguments:
  dataset               Dataset CSV path. If a relative path does not exist,
                        it is resolved under the data/ directory.

options:
  -h, --help            show this help message and exit
  -d, --D D             Restriction parameter value to test.
  -g, --gamma-mult GAMMA_MULT
                        Gamma multiplier of the theoretical gamma*.
  --alpha ALPHA         Subgradient alpha numerator used in alpha_k = alpha /
                        sqrt(k + 1).
  --eps EPS             Stopping tolerance.
  -i, --max-iter MAX_ITER
                        Maximum number of iterations per 


## `sda-bench.py` demo

`experiments/sda-bench.py` runs one registry objective through two solvers and saves one combined JSON summary. Objective functions available in `src/data/objectives.py`

Script output location:
`outputs/sda-bench/<objective>/results.json`



In [9]:
!python experiments/sda-bench.py -o max_affine_4d -d 10 -g 1.0 --alpha 1.0 -i 100 -r

SDA
objective_id: max_affine_4d
x*: [ 2.  , -1.5 ,  0.75,  2.5 ]
f*: 0.00000000
D      gamma/gamma*  converged  iterations  final_norm_gap  final_f(x_hat)  final_f(x)
10     1             False      100         0.57496271      0.25098355       0.08414076

Subgradient
objective_id: max_affine_4d
x*: [ 2.  , -1.5 ,  0.75,  2.5 ]
f*: 0.00000000
D      alpha      iterations  final_f(x_hat)  final_f(x)
10     1          100         0.15871441       0.07448918

Results JSON path: outputs/sda-bench/max_affine_4d/results.json



## Logistic-regression `sda-logreg.py`

The logistic loader expects:

- numeric feature columns,
- binary labels in `{0, 1}`,
- a label column such as `y`, `target`, `label`, `labels`, or `class`.

Output location:
`outputs/sda-logreg/<dataset-stem>/logreg/results.json`



### Script for generating logistic regression data

In [10]:
!python data/generate_logistic_data.py --help

usage: generate_logistic_data.py [-h] --n-samples N_SAMPLES
                                 --dimension DIMENSION --beta BETA [BETA ...]
                                 [--intercept INTERCEPT] [--seed SEED]
                                 [--flip-prob FLIP_PROB] [--output OUTPUT]

Generate synthetic binary logistic-regression data with labels in {0, 1}.

options:
  -h, --help            show this help message and exit
  --n-samples N_SAMPLES
                        Number of samples to generate.
  --dimension DIMENSION
                        Feature dimension.
  --beta BETA [BETA ...]
                        Coefficient vector. Its length must match --dimension.
  --intercept INTERCEPT
                        Intercept added to each logit.
  --seed SEED           Random seed.
  --flip-prob FLIP_PROB
                        Probability of independently flipping a generated
                        label.
  --output OUTPUT       Output CSV path. Default: /Users/michaliwaniuk/Desktop
 

In [11]:
!python data/generate_logistic_data.py --n-samples 300 --dimension 3 --beta 1.5 -2.0 0.75 --intercept -0.2 --seed 7 --output data/demo_logistic.csv
#inspect data in data folder

Saved synthetic logistic data to /Users/michaliwaniuk/Desktop/OIDA/data/demo_logistic.csv
Summary: n_samples=300, dimension=3, beta=[ 1.5 , -2.  ,  0.75], intercept=-0.200000, seed=7, positive_rate=0.4800


### Run `sda-logreg.py` on synthetic data and iris.csv

In [ ]:
!python experiments/sda-logreg.py data/demo_logistic.csv -d 2 -g 1 --alpha 0.5 -i 80

SDA
objective_id: demo_logistic_logreg
dataset: /Users/michaliwaniuk/Desktop/OIDA/data/demo_logistic.csv
lasso: False
train_samples: 240
test_samples: 60
method       solver       D      step_param    converged  iterations  runtime(s)  avg_iter(s)  train_loss  test_acc  nnz
sda          sda          2      1 gamma*      False      80          0.00110221  0.00001378   0.48889848  0.73333333 4

Subgradient
objective_id: demo_logistic_logreg
dataset: /Users/michaliwaniuk/Desktop/OIDA/data/demo_logistic.csv
lasso: False
train_samples: 240
test_samples: 60
method       solver       D      step_param    converged  iterations  runtime(s)  avg_iter(s)  train_loss  test_acc  nnz
subgradient  subgradient  2      alpha=0.5     False      80          0.00082442  0.00001031   0.45982831  0.73333333 4

sklearn
objective_id: demo_logistic_logreg
dataset: /Users/michaliwaniuk/Desktop/OIDA/data/demo_logistic.csv
lasso: False
train_samples: 240
test_samples: 60
method       solver       D      step_para

In [13]:
!python experiments/sda-logreg.py iris.csv -d 2 -g 1 --alpha 0.5 -i 80

SDA
objective_id: iris_logreg
dataset: /Users/michaliwaniuk/Desktop/OIDA/data/iris.csv
lasso: False
train_samples: 80
test_samples: 20
method       solver       D      step_param    converged  iterations  runtime(s)  avg_iter(s)  train_loss  test_acc  nnz
sda          sda          2      1 gamma*      False      80          0.00096150  0.00001202   0.15487599  1.00000000 5

Subgradient
objective_id: iris_logreg
dataset: /Users/michaliwaniuk/Desktop/OIDA/data/iris.csv
lasso: False
train_samples: 80
test_samples: 20
method       solver       D      step_param    converged  iterations  runtime(s)  avg_iter(s)  train_loss  test_acc  nnz
subgradient  subgradient  2      alpha=0.5     False      80          0.00065567  0.00000820   0.08853310  1.00000000 5

sklearn
objective_id: iris_logreg
dataset: /Users/michaliwaniuk/Desktop/OIDA/data/iris.csv
lasso: False
train_samples: 80
test_samples: 20
method       solver       D      step_param    converged  iterations  runtime(s)  avg_iter(s)  trai

## Generate plots
Inspect plots in outputs folder next to results.json for each run

In [14]:
!python outputs/generate_plots.py

Matplotlib is building the font cache; this may take a moment.
/Users/michaliwaniuk/Desktop/OIDA/outputs/generate_plots.py:388: UserWarning:

Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.

Generated plots for /Users/michaliwaniuk/Desktop/OIDA/outputs/sda-bench/max_affine_4d/results.json -> /Users/michaliwaniuk/Desktop/OIDA/outputs/sda-bench/max_affine_4d/plots/manifest.json
Generated plots for /Users/michaliwaniuk/Desktop/OIDA/outputs/sda-logreg/demo_logistic/logreg/results.json -> /Users/michaliwaniuk/Desktop/OIDA/outputs/sda-logreg/demo_logistic/logreg/plots/manifest.json
Generated plots for /Users/michaliwaniuk/Desktop/OIDA/outputs/sda-logreg/iris/logreg/results.json -> /Users/michaliwaniuk/Desktop/OIDA/outputs/sda-logreg/iris/logreg/plots/manifest.json
